In [1]:
# Attentionの仕組み
# seq2seqの問題点
# Encoderの改良
# Decoderの改良１

In [3]:
# Encoderが出力するhsと各単語の重みaから重み付き和を求める実装
import numpy as np

rng = np.random.default_rng()

T, H = 5, 4
hs = rng.random((T, H))
a = np.array([0.8, 0.1, 0.03, 0.05, 0.02])

ar = a.reshape(5, 1).repeat(4, axis=1)
print(ar.shape)

t = hs * ar
print(t.shape)

c = np.sum(t, axis=0)
print(c.shape)

(5, 4)
(5, 4)
(4,)


In [4]:
# バッチ処理版の重み付き和
N, T, H = 10, 5, 4
hs = rng.random((N, T, H))
a = rng.random((N, T))
ar = a.reshape(N, T, 1).repeat(H, axis=2)

t = hs * ar
print(t.shape)

c = np.sum(t, axis=1)
print(c.shape)

(10, 5, 4)
(10, 4)


In [5]:
# Weight Sumレイヤの実装
class WeightSum:
    def __init__(self):
        self.params, self.grads = [], []
        self.cache = None

    def forward(self, hs, a):
        N, T, H = hs.shape

        ar = a.reshape(N, T, 1).repeat(H, axis=2)
        t = hs * ar
        c = np.sum(t, axis=1)

        self.cache = (hs, ar)
        return c

    def backward(self, dc):
        hs, ar = self.cache
        N, T, H = hs.shape

        dt = dc.reshape(N, 1, H).repeat(T, axis=1)
        dar = dt* hs
        dhs = dt * ar
        da = np.sum(dar, axis=2)

        return dhs, da

In [6]:
# Decoderの改良２
# Softmax関数を適用する
import sys
sys.path.append("..")
from common.layers import Softmax
import numpy as np

N, T, H = 10, 5, 4
hs = rng.random((N, T, H))
h = rng.random((N, H))
hr = h.reshape(N, 1, H).repeat(T, axis=1)

t = hs * hr
print(t.shape)

s = np.sum(t, axis=2)
print(s.shape)

softmax = Softmax()
a = softmax.forward(s)
print(a.shape)

(10, 5, 4)
(10, 5)
(10, 5)
